# rlm-gateway via OpenRouter's free tier

Simpler than the Ollama/GPU version -- OpenRouter is a hosted API, so there's no install, no GPU, no context-window tuning, nothing that can get wiped by a runtime reset. Get a free key first at **openrouter.ai/keys** (no card required).

Rate limit on an unfunded account: **50 requests/day**. Each case can use several requests (one per RLM iteration), so this is for testing a handful of cases, not the full 375.

## 1. Install dependencies

In [ ]:
!pip install -q huggingface-hub pandas pyarrow python-dotenv rlms

## 2. Set your OpenRouter API key

Typed via a hidden prompt so it never gets saved in plain text in this notebook file.

In [ ]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

## 3. Write out the rlm_gateway package

Same code as `src/rlm_gateway/` in the project, copied in directly so this notebook is self-contained.

In [ ]:
os.makedirs("rlm_gateway", exist_ok=True)
open("rlm_gateway/__init__.py", "w").close()

In [ ]:
%%writefile rlm_gateway/dataset.py
"""
RCAEval dataset access: download, case-id parsing, and per-case loading.

RCAEval's RE1 (metrics-only) suite has no separate index table -- each case
is a folder named "re1{system}_{service}_{fault}_{run}" containing
metrics.parquet (one row per timestamp, one column per "{service}_{metric}"
pair) and inject_time.txt (a single unix timestamp). The ground-truth root
cause is encoded directly in the folder name.
"""

import os
import re
from dataclasses import dataclass

import pandas as pd
from huggingface_hub import snapshot_download

DATASET_REPO_ID = "phamquiluan/RCAEval"

# Fault types are spelled out explicitly (rather than matched greedily)
# because service names can themselves contain hyphens, e.g. "ts-auth-service".
CASE_NAME_RE = re.compile(
    r"^re1(?P<system>[a-z]+)_(?P<service>.+)_(?P<fault>cpu|mem|disk|delay|loss)_(?P<run>\d+)$"
)

SYSTEM_NAMES = {"ob": "online-boutique", "ss": "sock-shop", "tt": "train-ticket"}


@dataclass(frozen=True)
class Case:
    case_id: str
    system: str
    root_cause_service: str
    fault_type: str
    run: int


def download_suite(pattern: str = "re1*") -> str:
    """Download a suite of RCAEval cases and return the local snapshot path."""
    return snapshot_download(repo_id=DATASET_REPO_ID, repo_type="dataset", allow_patterns=pattern)


def parse_case_id(case_id: str) -> Case:
    match = CASE_NAME_RE.match(case_id)
    if not match:
        raise ValueError(f"Unrecognized case id: {case_id!r}")
    return Case(
        case_id=case_id,
        system=SYSTEM_NAMES.get(match["system"], match["system"]),
        root_cause_service=match["service"],
        fault_type=match["fault"],
        run=int(match["run"]),
    )


def list_cases(local_path: str) -> list[Case]:
    return sorted(
        (parse_case_id(name) for name in os.listdir(local_path) if CASE_NAME_RE.match(name)),
        key=lambda c: c.case_id,
    )


def load_case_metrics(local_path: str, case_id: str) -> pd.DataFrame:
    return pd.read_parquet(os.path.join(local_path, case_id, "metrics.parquet"))


def load_inject_time(local_path: str, case_id: str) -> int:
    with open(os.path.join(local_path, case_id, "inject_time.txt")) as f:
        return int(f.read().strip())

In [ ]:
%%writefile rlm_gateway/tools.py
"""
Custom tools exposed to the RLM root model's REPL (via RLM's
`custom_tools={...}` argument). Each metrics.parquet column is named
"{service}_{metric}", e.g. "carts-db_cpu" or "front-end_latency-90" --
these helpers split on that convention so the model can explore metrics
by service instead of getting all 50-60 raw columns dumped in at once.

The three RCAEval systems don't share one metric vocabulary: sock-shop and
train-ticket use "_workload"/"_latency-50"/"_latency-90", while
online-boutique uses "_load"/"_latency" instead. Both are matched here.
"""

import pandas as pd

# Longest/most specific suffixes first so e.g. "_latency-50" isn't mistaken
# for ending in "_50", and "_latency-50"/"_latency-90" aren't mistaken for
# ending in the shorter "_latency".
METRIC_SUFFIXES = [
    "_latency-50",
    "_latency-90",
    "_workload",
    "_latency",
    "_load",
    "_error",
    "_cpu",
    "_mem",
]


def split_metric_column(column: str) -> tuple[str, str]:
    for suffix in METRIC_SUFFIXES:
        if column.endswith(suffix):
            return column[: -len(suffix)], suffix[1:]
    raise ValueError(f"Unrecognized metric column: {column!r}")


def list_services(metrics_df: pd.DataFrame) -> list[str]:
    services = {split_metric_column(col)[0] for col in metrics_df.columns if col != "time"}
    return sorted(services)


def list_categories(metrics_df: pd.DataFrame) -> list[str]:
    """The metric categories actually present in this case (varies by system)."""
    categories = {split_metric_column(col)[1] for col in metrics_df.columns if col != "time"}
    return sorted(categories)


def make_query_service_tool(metrics_df: pd.DataFrame):
    """Build a query_service(service_name, category=None) tool bound to one case's metrics."""

    def query_service(service_name: str, category: str | None = None) -> dict:
        """
        Return {"time": [...], "<column>": [...], ...} for one service.
        category, if given, filters to one of: cpu, mem, workload, error,
        latency-50, latency-90.
        """
        matched = [
            col
            for col in metrics_df.columns
            if col != "time"
            and split_metric_column(col)[0] == service_name
            and (category is None or split_metric_column(col)[1] == category)
        ]
        if not matched:
            return {"error": f"No metrics for service={service_name!r} category={category!r}"}
        return metrics_df[["time", *matched]].to_dict(orient="list")

    return query_service

In [ ]:
%%writefile rlm_gateway/_rlm_compat.py
"""
Workaround for a bug in rlms==0.1.3's AnthropicClient: it assumes
response.content[0] is always a text block, but Claude returns a
ThinkingBlock first when extended thinking is on, which has no .text
attribute. Patches completion()/acompletion() to pick out the first
real text block instead. Not needed for the openrouter backend, but
pipeline.py imports it unconditionally, so it must exist.
"""

from rlm.clients import anthropic as rlm_anthropic


def _first_text(content) -> str:
    for block in content:
        if getattr(block, "type", None) == "text":
            return block.text
    return ""


def completion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    if not model:
        raise ValueError("Model name is required for Anthropic client.")
    kwargs = {"model": model, "max_tokens": self.max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = self.client.messages.create(**kwargs)
    self._track_cost(response, model)
    return _first_text(response.content)


async def acompletion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    if not model:
        raise ValueError("Model name is required for Anthropic client.")
    kwargs = {"model": model, "max_tokens": self.max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = await self.async_client.messages.create(**kwargs)
    self._track_cost(response, model)
    return _first_text(response.content)


rlm_anthropic.AnthropicClient.completion = completion
rlm_anthropic.AnthropicClient.acompletion = acompletion

In [ ]:
%%writefile rlm_gateway/pipeline.py
"""
Builds an RLM instance with query_service/list_services as custom tools,
runs it on one RCAEval case, and checks the prediction against ground truth.
"""

import os

from rlm import RLM

from . import _rlm_compat  # noqa: F401  (patches AnthropicClient's thinking-block bug)
from .dataset import Case, download_suite, load_case_metrics, load_inject_time
from .tools import list_categories, list_services, make_query_service_tool

PROMPT_TEMPLATE = """\
You are diagnosing a microservice incident. A fault was injected at unix \
timestamp {inject_time} into the "{system}" system, which has these \
services: {services}.

You have two tools:
- list_services() -> list[str]
- query_service(service_name, category=None) -> dict with a "time" key and \
one key per metric column. category is one of: {categories}.

Each query can return hundreds of raw data points. Do NOT print raw \
query_service() results directly -- compute and print a summary instead \
(e.g. mean/min/max before vs. after the injection timestamp).

Use them to inspect metrics around the injection time and figure out which \
service is the root cause of the incident. Respond with ONLY the service \
name as your final answer.
"""


# Free tier: 50 requests/day on an unfunded OpenRouter account. qwen3-coder
# is benchmarked as OpenRouter's strongest free option for tool-calling/
# agentic tasks specifically.
BACKEND_CONFIGS = {
    "anthropic": {"rlm_backend": "anthropic", "api_key_env": "ANTHROPIC_API_KEY", "default_model": "claude-sonnet-5"},
    "openai": {"rlm_backend": "openai", "api_key_env": "OPENAI_API_KEY", "default_model": "gpt-4o-mini"},
    "openrouter": {
        "rlm_backend": "openrouter",
        "api_key_env": "OPENROUTER_API_KEY",
        "default_model": "qwen/qwen3-coder:free",
    },
}


def run_case(
    local_path: str,
    case: Case,
    backend: str = "openrouter",
    model_name: str | None = None,
    verbose: bool = True,
) -> str:
    metrics_df = load_case_metrics(local_path, case.case_id)
    inject_time = load_inject_time(local_path, case.case_id)
    services = list_services(metrics_df)
    categories = list_categories(metrics_df)

    config = BACKEND_CONFIGS[backend]
    model_name = model_name or config["default_model"]
    api_key = os.getenv(config["api_key_env"]) if config["api_key_env"] else "ollama"

    backend_kwargs = {"model_name": model_name, "api_key": api_key}
    if "base_url" in config:
        backend_kwargs["base_url"] = config["base_url"]

    rlm = RLM(
        backend=config["rlm_backend"],
        backend_kwargs=backend_kwargs,
        environment="local",
        custom_tools={
            "list_services": {
                "tool": lambda: services,
                "description": "List every service name present in this case's metrics.",
            },
            "query_service": {
                "tool": make_query_service_tool(metrics_df),
                "description": (
                    "Fetch metrics for one service, optionally filtered to a "
                    "category (cpu, mem, workload, error, latency-50, latency-90)."
                ),
            },
        },
        verbose=verbose,
    )

    prompt = PROMPT_TEMPLATE.format(
        inject_time=inject_time,
        system=case.system,
        services=", ".join(services),
        categories=", ".join(categories),
    )
    result = rlm.completion(prompt)
    return result.response.strip()

## 4. Run one real case

In [ ]:
from rlm_gateway.dataset import Case, download_suite
from rlm_gateway.pipeline import run_case

local_path = download_suite(pattern="re1ss_carts_mem_4/*")
case = Case(
    case_id="re1ss_carts_mem_4",
    system="sock-shop",
    root_cause_service="carts",
    fault_type="mem",
    run=4,
)

prediction = run_case(local_path, case, backend="openrouter")
print(f"\nPredicted: {prediction}")
print(f"Actual:    {case.root_cause_service}")
print(f"Correct:   {case.root_cause_service in prediction}")

## 5. Small batch

Keep `LIMIT` small -- 50 requests/day total on an unfunded account, and each case can use several requests.

In [ ]:
import csv, time
from rlm_gateway.dataset import list_cases

LIMIT = 5

local_path = download_suite(pattern="re1*")
cases = list_cases(local_path)[:LIMIT]

with open("eval_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["case_id", "system", "fault_type", "actual", "predicted", "correct", "elapsed_s", "error"])
    writer.writeheader()
    correct_count = 0
    for i, case in enumerate(cases, 1):
        print(f"[{i}/{len(cases)}] {case.case_id} (actual={case.root_cause_service}) ... ", end="", flush=True)
        start = time.time()
        try:
            prediction = run_case(local_path, case, backend="openrouter", verbose=False)
            correct = case.root_cause_service in prediction
            error = None
        except Exception as exc:
            prediction, correct, error = None, False, str(exc)
        elapsed = time.time() - start
        correct_count += correct
        print(f"predicted={prediction} correct={correct} ({elapsed:.1f}s)")
        writer.writerow({"case_id": case.case_id, "system": case.system, "fault_type": case.fault_type,
                          "actual": case.root_cause_service, "predicted": prediction, "correct": correct,
                          "elapsed_s": round(elapsed, 1), "error": error or ""})
        f.flush()

print(f"\nAccuracy: {correct_count}/{len(cases)} ({correct_count/len(cases):.1%})")

In [ ]:
from google.colab import files
files.download("eval_results.csv")